Config

In [ ]:
setup = """
set -e

sudo apt update
sudo apt install -y build-essential git wget ca-certificates

git clone https://github.com/Decrescent398/arc-agi-3-solution.git /repo
cd /repo/ARC3-Inference
chmod +x scripts/*.sh

uv python install 3.12.12
uv venv --python 3.12.12 .venv
uv lock --python 3.12.12
make install

cd /tmp
wget https://developer.download.nvidia.com/compute/cuda/12.8.1/local_installers/cuda_12.8.1_570.124.06_linux.run
sudo sh cuda_12.8.1_570.124.06_linux.run --silent --toolkit --toolkitpath=/opt/cuda-12.8 --override
test -x /opt/cuda-12.8/bin/nvcc
/opt/cuda-12.8/bin/nvcc --version

uv pip install --python /repo/ARC3-Inference/.venv/bin/python nvidia-cuda-runtime-cu12==12.8.* nvidia-cuda-nvcc-cu12==12.8.* nvidia-cuda-nvrtc-cu12==12.8.* nvidia-cuda-cupti-cu12==12.8.* nvidia-cuda-cccl-cu12

export CUDA_HOME=/opt/cuda-12.8
export CUDA_PATH=/opt/cuda-12.8
export VIRTUAL_ENV=/repo/ARC3-Inference/.venv

export PATH="/opt/cuda-12.8/bin:$VIRTUAL_ENV/bin:$PATH"
export LD_LIBRARY_PATH=/opt/cuda-12.8/lib64
export LIBRARY_PATH=/opt/cuda-12.8/lib64

CCCL="$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cuda_cccl/include"

test -d "$CCCL"
mkdir -p /tmp/arc3-cuda-includes
ln -sfn "$CCCL" /tmp/arc3-cuda-includes/cccl

export CPATH="/opt/cuda-12.8/include:/tmp/arc3-cuda-includes:$CCCL"
"""

Setup

In [ ]:
import os
import subprocess
from pathlib import Path

In [ ]:
import marimo as mo
import json
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
venv = Path("/repo/ARC3-Inference/.venv")
cuda = Path("/opt/cuda-12.8")
cccl = venv / "lib/python3.12/site-packages/nvidia/cuda_cccl/include"

installed = all([
    Path("/repo/ARC3-Inference").exists(),
    (venv / "bin/python").exists(),
    (venv / "bin/uv").exists() or (venv / "bin/python").exists(),
    (cuda / "bin/nvcc").exists()
])

if not installed:
    subprocess.run(["bash", "-c", setup], check=True)
else:
    print("Setup already installed. Skipping.")

compat = Path("/tmp/arc3-cuda-includes")
compat.mkdir(parents=True, exist_ok=True)
link = compat / "cccl"

if link.is_symlink() or link.is_file():
    link.unlink()
if not link.exists():
    link.symlink_to(cccl, target_is_directory=True)

env = os.environ.copy()
env.update({
    "VIRTUAL_ENV": str(venv),
    "CUDA_HOME": str(cuda),
    "CUDA_PATH": str(cuda),
    "PATH": f"{venv}/bin:{cuda}/bin:{env['PATH']}",
    "CPATH": f"{cuda}/include:{compat}:{cccl}",
    "LIBRARY_PATH": f"{cuda}/lib64",
    "LD_LIBRARY_PATH": f"{cuda}/lib64",
})

mo.output.clear_console()

In [ ]:
try:
    os.mkdir("results")
except FileExistsError:
    pass

In [ ]:
def sh(process: str, env: dict=env):
    try:
        subprocess.run(process, check=True, shell=True, env=env)
    except (TypeError, ValueError, subprocess.CalledProcessError):
        exit()
    return True

In [ ]:
D = "/repo/ARC3-Inference"
GAME = "tn36-ef4dde99"
N_PASSES = 1
MAX_RUNTIME_MINUTES = 10
ENVIRONMENTS_DIR = "/results/envs"
EXPERIMENTS_DIR= "/results"

In [ ]:
start_server = f"cd {D} && uv run --no-sync --python 3.12.12 python -c \"import arc_agi; arc_agi.Arcade(environments_dir='{ENVIRONMENTS_DIR}').make('{GAME}')\""
server = sh(start_server)

Execute

```console
tail -f /repo/ARC3-Inference/.cache/arc3_runtime/arc3-inference-server.log
```

In [ ]:
run = f"cd {D} && make server && make interactive GAME={GAME} N_PASSES={N_PASSES} " \
      f"MAX_RUNTIME_MINUTES={MAX_RUNTIME_MINUTES} ENVIRONMENTS_DIR={ENVIRONMENTS_DIR} EXPERIMENTS_DIR={EXPERIMENTS_DIR}"
      
list_games = f"cd {D} && uv run --no-sync inference-taaf-run --include-tags official --list-games"

reasoning_traces = f"uv run --no-sync inference-traces --runs-dir {EXPERIMENTS_DIR}"

In [ ]:
sh(run)
sh(reasoning_traces)

Results

In [ ]:
game_path = max(Path(EXPERIMENTS_DIR).rglob("*_events.jsonl"), key=lambda p: p.stat().st_mtime)
traces_path = list((Path(EXPERIMENTS_DIR) / "traces").rglob("*.json"))

events = [json.loads(line) for line in game_path.read_text().splitlines()]
frames = [e for e in events if isinstance(e.get("board"), list)]
traces = [json.loads(f.read_text()) for f in traces_path]

# step = mo.ui.slider(start=0, stop=len(frames)-1, label="step", show_value=True)
get_step, set_step = mo.state(0)
nxt = mo.ui.button(
    on_click=lambda _: set_step(get_step() + 1) if get_step() < len(frames) else set_step(get_step()), 
    label="next"
)
prev = mo.ui.button(
    on_click=lambda _: set_step(get_step() - 1) if get_step() > 0 else set_step(get_step()), 
    label="prev"
)

In [ ]:
step = get_step()
event = frames[step]
trace = traces[0]["messages"][step]
 
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.array(event["board"]), interpolation="nearest", vmin=0, vmax=15)
ax.axis("off")
plt.close(fig)

actions = mo.md(
    f"**ACTION:** {event.get('action_display', 'RESET')} " \
    f"**SCORE:** {event.get('score', 0)} " \
    f"**LEVEL:** {event.get('level', 1)} <br> <br>" \
    f"**REASONING:** {trace.get('reasoning', '')}"
)

mo.vstack([
    mo.hstack([prev, mo.md(f"**Step:**{step}/{len(frames)}"), nxt], gap=2, justify="start"),
    actions,
    mo.as_html(fig)
])